# Notebook 3 — Symbolic PPO: Train on World 1-1

Trains a **PPO** agent on World 1-1 using the RAM-based symbolic grid. Compared to Notebook 1 (DQN), PPO typically achieves stronger and more stable performance. Compared to Notebook 2 (Pixel PPO), the symbolic observation is much lower-dimensional — no CNN needed.

## Study Progression
1. Symbolic DQN — World 1-1
2. Pixel PPO — World 1-1
3. **→ Symbolic PPO — World 1-1 (you are here)**
4. Transfer Learning — Symbolic PPO 1-1 → 1-2
5. World 1 Full — Random Starts + All 4 Levels

## Setup
- **Observation:** flattened 13×16 symbolic grid × 4 frame-stack + power state = **833-dim** vector
- **Policy:** `MlpPolicy` with `[512, 512]` hidden layers
- **Training:** 8 parallel envs, two phases (2M + 2M steps), faster than pixel (no CNN/image processing)
- **Output:** `models/symbolic_ppo/`

In [ ]:
# --- Google Colab Setup ---
import os, sys

if os.getenv('COLAB_RELEASE_TAG'):
    !pip install -q Cython setuptools wheel
    !git clone -b hotfix/version1 https://github.com/lmartim4/CSC-52081-ContinousMountainCar.git /content/repo
    %cd /content/repo
    !pip install -r requirements.txt --no-build-isolation
    sys.path.insert(0, '/content/repo')
    import site; SITE = site.getsitepackages()[0]
    !patch --forward -p0 {SITE}/nes_py/_rom.py                   < patches/nes_py_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym/utils/passive_env_checker.py < patches/gym_bool8_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym_super_mario_bros/smb_env.py  < patches/smb_env_numpy2.patch || true
    !sed -i 's/observation, reward, terminated, truncated, info = self.env.step(action)/_result = self.env.step(action); observation, reward, terminated, info = _result[:4]; truncated = _result[4] if len(_result) > 4 else False/' {SITE}/gym/wrappers/time_limit.py
    !git pull
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        %cd ..

In [ ]:
import torch
from stable_baselines3 import PPO

from src.wrappers import make_symbolic_vec_env, make_symbolic_env
from src.utils.callbacks import CheckpointAndLogCallback
from src.config import PPOConfig

print(f'Using CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Create 8 parallel symbolic environments
NUM_ENVS = 8

env = make_symbolic_vec_env(
    env_id='SuperMarioBros-1-1-v3',
    skip=4,
    n_stack=4,
    flatten=True,
    num_envs=NUM_ENVS,
)
print(f'Observation space: {env.observation_space.shape}')
print(f'Action space: {env.action_space.n}')
print(f'Parallel envs: {NUM_ENVS}')

In [ ]:
# Phase 1: Train from scratch
config = PPOConfig(
    policy='MlpPolicy',
    learning_rate=2.5e-4,
    n_steps=512,
    batch_size=256,
    n_epochs=4,
    gamma=0.9,           # vietnh1009 uses 0.9
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
    total_timesteps=2_000_000,
)

model = PPO(
    policy=config.policy,
    env=env,
    learning_rate=config.learning_rate,
    n_steps=config.n_steps,
    batch_size=config.batch_size,
    n_epochs=config.n_epochs,
    gamma=config.gamma,
    gae_lambda=config.gae_lambda,
    clip_range=config.clip_range,
    ent_coef=config.ent_coef,
    vf_coef=config.vf_coef,
    max_grad_norm=config.max_grad_norm,
    policy_kwargs=dict(net_arch=[512, 512]),
    tensorboard_log='logs/symbolic_ppo',
    verbose=1,
    device='cpu',  # MLP is faster on CPU
)

print(f'Phase 1: {config.total_timesteps:,} timesteps')
print(f'Device: {model.device}')
print(f'Batch per rollout: {config.n_steps * NUM_ENVS} samples')

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/symbolic_ppo

In [ ]:
# Phase 1: Train for 2M steps
callback = CheckpointAndLogCallback(
    save_path='models/symbolic_ppo',
    save_freq=50_000,
)

model.learn(
    total_timesteps=config.total_timesteps,
    callback=callback,
    log_interval=1,
)
model.save('models/symbolic_ppo/phase1_model')
print('Phase 1 complete!')

In [ ]:
# Phase 2: Load best model, lower lr, train 2M more
from stable_baselines3 import PPO

model = PPO.load('models/symbolic_ppo/phase1_model', env=env, device='cpu')
model.learning_rate = 1e-5

callback2 = CheckpointAndLogCallback(
    save_path='models/symbolic_ppo',
    save_freq=50_000,
)

model.learn(
    total_timesteps=2_000_000,
    callback=callback2,
    log_interval=1,
)
model.save('models/symbolic_ppo/final_model')
print('Phase 2 complete!')

In [ ]:
# Plot training results
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

rewards = callback.episode_rewards + (callback2.episode_rewards if 'callback2' in dir() else [])
lengths = callback.episode_lengths + (callback2.episode_lengths if 'callback2' in dir() else [])
flags = callback.episode_flags + (callback2.episode_flags if 'callback2' in dir() else [])

if len(rewards) > 0:
    window = min(100, len(rewards))

    ax = axes[0]
    ax.plot(rewards, alpha=0.3, color='blue')
    if len(rewards) > window:
        smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
        ax.plot(range(window-1, len(rewards)), smoothed, color='blue', linewidth=2)
    ax.set_title('Episode Rewards')
    ax.set_xlabel('Episode')

    ax = axes[1]
    ax.plot(lengths, alpha=0.3, color='orange')
    if len(lengths) > window:
        smoothed = np.convolve(lengths, np.ones(window)/window, mode='valid')
        ax.plot(range(window-1, len(lengths)), smoothed, color='orange', linewidth=2)
    ax.set_title('Episode Lengths')
    ax.set_xlabel('Episode')

    ax = axes[2]
    if len(flags) > 0:
        cumulative_flags = np.cumsum(flags) / (np.arange(len(flags)) + 1)
        ax.plot(cumulative_flags, color='green', linewidth=2)
    ax.set_title('Cumulative Flag Rate')
    ax.set_xlabel('Episode')
    ax.set_ylim(0, 1)

    plt.tight_layout()
    plt.savefig('models/symbolic_ppo/training_curves.png', dpi=150)
    plt.show()
else:
    print('No episode data collected yet.')

In [ ]:
# Evaluate final model
from stable_baselines3 import PPO
import numpy as np

eval_model = PPO.load('models/symbolic_ppo/final_model')

eval_env = make_symbolic_env(
    env_id='SuperMarioBros-1-1-v3',
    skip=4, n_stack=4, flatten=True,
)

rewards = []
lengths = []
flags = []

for ep in range(10):
    reset_result = eval_env.reset()
    obs = reset_result[0] if isinstance(reset_result, tuple) else reset_result
    done = False
    total_reward = 0.0
    steps = 0
    flag = False

    while not done:
        action, _ = eval_model.predict(obs, deterministic=True)
        step_result = eval_env.step(int(action))
        if len(step_result) == 5:
            obs, reward, terminated, truncated, info = step_result
            done = terminated or truncated
        else:
            obs, reward, done, info = step_result
        total_reward += float(reward)
        steps += 1
        if isinstance(info, dict) and info.get('flag_get', False):
            flag = True

    rewards.append(total_reward)
    lengths.append(steps)
    flags.append(flag)
    status = 'FLAG!' if flag else 'DEAD'
    print(f'Episode {ep+1}: reward={total_reward:.1f}, steps={steps}, {status}')

print(f'\nMean reward: {np.mean(rewards):.1f} +/- {np.std(rewards):.1f}')
print(f'Mean length: {np.mean(lengths):.0f}')
print(f'Flag rate: {np.mean(flags):.0%}')

eval_env.close()